# 18BI · Hello world — is your machine ready?

**Run every cell, top to bottom.** It takes a few seconds and changes nothing on your computer.
At the end you get either

```
ALL GOOD — you are ready for week 2.
```

or a short list of what is missing.

### Before you start

From the repository root — the folder with `pyproject.toml` — in a terminal:

```bash
uv sync
uv run jupyter lab
```

`uv sync` builds the project's own Python environment in `.venv`; `uv run` starts Jupyter
inside it. Open this notebook in the Jupyter tab that appears in your browser.

No `uv`? `python -m venv .venv && .venv/bin/pip install -r requirements.txt`, then
`.venv/bin/jupyter lab`.

## 1 · Which Python is this?

The most common problem in this course is having two Pythons and running the wrong one.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "exercises" else Path.cwd()
in_project_venv = str(ROOT / ".venv") in sys.executable

print("python              ", ".".join(map(str, sys.version_info[:3])))
print("running from        ", sys.executable)
print("the project's .venv?", "yes" if in_project_venv else "NO -- see below")

checks = {
    "Python 3.10 or newer": sys.version_info >= (3, 10),
    "running in the project's .venv": in_project_venv,
}

> **"NO"** means this notebook runs on a different Python than the one `uv sync` set up —
> usually a leftover from an earlier Anaconda or Jupyter installation. Register the project's
> environment as a kernel, once per machine:
>
> ```bash
> uv run python -m ipykernel install --user --name 18bi --display-name "18BI (uv)"
> ```
>
> then restart Jupyter, choose **Kernel → Change Kernel… → 18BI (uv)**, and run this notebook
> again.

## 2 · The packages

In [ ]:
for name in ["duckdb", "pandas", "matplotlib"]:
    try:
        module = __import__(name)
        print(f"  ok       {name:<11} {module.__version__}")
        checks[f"{name} installed"] = True
    except ImportError:
        print(f"  MISSING  {name}")
        checks[f"{name} installed"] = False

## 3 · A first query

DuckDB is the database we will use all semester. It needs no server and no account, and it
can read a CSV file as if it were a table — here, the seven order lines from week 1's handout.

In [ ]:
import duckdb

orders = ROOT / "data" / "handout" / "orders.csv"
duckdb.sql(f"select order_id, product, quantity, unit_price_net, status from '{orders}'").show()

checks["DuckDB reads the handout"] = duckdb.sql(f"select count(*) from '{orders}'").fetchone()[0] == 7

## 4 · Verdict

In [ ]:
for name, ok in checks.items():
    print(f"  {'ok  ' if ok else 'FAIL'}  {name}")

print()
if all(checks.values()):
    print("ALL GOOD — you are ready for week 2.")
else:
    print("Not there yet — see the notes below the cell that failed.")

---

### If something failed

**"the project's .venv? NO"** — register the kernel as described in section 1.

**A package is MISSING** — run `uv sync` again from the repository root and check that it
finishes without errors, then restart the kernel.

**The handout cannot be read** — run `git pull`; the file is `data/handout/orders.csv`.

**`uv: command not found`** — install it from [docs.astral.sh/uv](https://docs.astral.sh/uv/),
or use the `pip` route at the top of this notebook.

Still stuck after ten minutes? Stop, and show me the output — that is quicker than guessing.